In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

from src.utils.logger import setup_logger
from src.utils.paths import RunPaths

PIPELINE_NAME = "beat-mask"
RUN_TRAINING = False
RESUME_RUN_ID = None

OUTPUT_DIR = ROOT / "output" if RUN_TRAINING else ROOT / "output" / "test"
run = (
    RunPaths.from_run_id(RESUME_RUN_ID, OUTPUT_DIR) if RESUME_RUN_ID else RunPaths.new(PIPELINE_NAME, OUTPUT_DIR)
).create()

logger = setup_logger(run.run_id, run.log_file)
logger.info("pipeline=%s | run=%s | mode=%s", PIPELINE_NAME, run.run_id, "TRAINING" if RUN_TRAINING else "SMOKE TEST")
logger.info(
    "log=%s | checkpoint=%s | result=%s | figure=%s", run.log_file, run.checkpoint_dir, run.result_dir, run.figure_dir
)

In [ ]:
from dataclasses import replace

import torch
import yaml

from src.train import BEATTrainConfig
from src.utils.seed import set_seed

with open(ROOT / "config" / "beat.yaml", encoding="utf-8") as _f:
    beat_yaml = yaml.safe_load(_f)

ALL_SNR = (-5, 0, 5, 10, 15, 20)

CONFIG = {
    "seed": beat_yaml["train"]["seed"],
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "num_workers": 4,
    "resume": RESUME_RUN_ID is not None,
    "max_samples": None,
    "sample_rate": beat_yaml["frontend"]["sample_rate"],
    "clip_seconds": 4.0,
    "snr_groups": {
        "low_snr": (-5, 0),
        "mid_snr": (5, 10),
        "high_snr": (15, 20),
        "all_snr": ALL_SNR,
    },
    "maskings": (True,),
    "beat": BEATTrainConfig.from_dict({**beat_yaml["train"], "label_smoothing": beat_yaml["loss"]["label_smoothing"]}),
}

if not RUN_TRAINING:
    CONFIG.update(max_samples=8, num_workers=0)
    CONFIG["beat"] = replace(CONFIG["beat"], epochs=1, batch_size=4, eval_batch_size=4)

DEVICE = torch.device(CONFIG["device"])
set_seed(CONFIG["seed"])
logger.info(
    "run_training=%s | device=%s | seed=%d | resume=%s | max_samples=%s",
    RUN_TRAINING,
    DEVICE,
    CONFIG["seed"],
    CONFIG["resume"],
    CONFIG["max_samples"],
)
logger.info(
    "snr_groups=%s | beat epochs=%d patience=%d",
    CONFIG["snr_groups"],
    CONFIG["beat"].epochs,
    CONFIG["beat"].patience,
)

In [ ]:
from scripts.download_dataset import ensure_dataset
from scripts.load_dataset import MIXTURE_KEY, Mixture36Dataset, build_dataloader, subsample

ensure_dataset(MIXTURE_KEY, ROOT / "datasets")

max_samples = CONFIG["max_samples"]


def make_loader(dataset, batch_size, shuffle):
    return build_dataloader(dataset, batch_size, shuffle, CONFIG["num_workers"])


def make_beat_loaders(snr_levels):
    beat_config = CONFIG["beat"]
    datasets = {
        split: subsample(
            Mixture36Dataset(split=split, snr_db=tuple(float(s) for s in snr_levels), duration=CONFIG["clip_seconds"]),
            max_samples,
        )
        for split in ("train", "validation", "test")
    }
    return {
        "train": make_loader(datasets["train"], beat_config.batch_size, True),
        "validation": make_loader(datasets["validation"], beat_config.eval_batch_size, False),
        "test": make_loader(datasets["test"], beat_config.eval_batch_size, False),
    }


for _levels in CONFIG["snr_groups"].values():
    _sizes = {split: len(loader.dataset) for split, loader in make_beat_loaders(_levels).items()}
    logger.info("%s (beat) snr=%s: %s", MIXTURE_KEY, _levels, _sizes)

In [ ]:
from scripts.load_pretrain import PRETRAINED, download_pretrained

FRCRN_CHECKPOINT = ROOT / "pretrained" / "frcrn_best.pt"
if not FRCRN_CHECKPOINT.is_file():
    raise FileNotFoundError(f"put the fine-tuned FRCRN checkpoint at {FRCRN_CHECKPOINT}")

logger.info("downloading / locating pretrained checkpoints")
pretrained_paths = {"beat": download_pretrained("beat"), "frcrn": FRCRN_CHECKPOINT}
for _name, _path in pretrained_paths.items():
    logger.info("%s pretrain: %s (%.1f MB)", _name, _path.name, _path.stat().st_size / 1e6)

In [ ]:
beat_spec, frcrn_spec = PRETRAINED["beat"], PRETRAINED["frcrn"]

beat_pretrained = beat_spec.model_cls.from_yaml(beat_spec.config_path)
beat_report = beat_pretrained.load_pretrained(pretrained_paths["beat"])
logger.info("beat: pretrained loaded | missing=%d unexpected=%d", len(beat_report.missing), len(beat_report.unexpected))

separator = frcrn_spec.model_cls.from_yaml(frcrn_spec.config_path)
frcrn_report = separator.load_pretrained(pretrained_paths["frcrn"])
logger.info(
    "frcrn: pretrained loaded | missing=%d unexpected=%d | target=%s",
    len(frcrn_report.missing),
    len(frcrn_report.unexpected),
    separator.config.target,
)

beat_state = {key: value.clone() for key, value in beat_pretrained.state_dict().items()}


def new_beat():
    model = beat_spec.model_cls.from_yaml(beat_spec.config_path)
    model.load_state_dict(beat_state)
    return model


logger.info(
    "params: frcrn=%.2fM | beat=%.2fM", separator.num_parameters() / 1e6, beat_pretrained.num_parameters() / 1e6
)

In [ ]:
import csv
import json

from tqdm.auto import tqdm

from src.train import BEATTrainer

metric_names = ["top1_accuracy", "top3_accuracy", "balanced_accuracy", "macro_f1", "mAP", "macro_auc"]


def run_scenario(index, total, group, levels, masking):
    name = f"{group}_{'mask' if masking else 'nomask'}"
    result_path = run.history_path(name)
    if CONFIG["resume"] and result_path.is_file():
        logger.info("[%d/%d] %s: loaded cached result", index, total, name)
        return json.loads(result_path.read_text(encoding="utf-8"))

    logger.info("[%d/%d] %s: snr=%s masking=%s", index, total, name, levels, masking)
    set_seed(CONFIG["seed"])
    loaders = make_beat_loaders(levels)
    trainer = BEATTrainer(
        new_beat(),
        CONFIG["beat"],
        DEVICE,
        logger,
        name,
        run.checkpoint_path(name),
        separator if masking else None,
    )
    fit = trainer.fit(loaders["train"], loaders["validation"])
    test = trainer.evaluate(loaders["test"], desc=f"{name} - test")
    result = {
        "scenario": name,
        "snr_group": group,
        "snr_levels": list(levels),
        "masking": masking,
        "best_epoch": fit.best_epoch,
        "epochs_run": fit.epochs_run,
        "test": test,
        "history": fit.history,
    }
    result_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    logger.info(
        "[%d/%d] %s: test top1=%.4f macro_f1=%.4f mAP=%.4f",
        index,
        total,
        name,
        test["top1_accuracy"],
        test["macro_f1"],
        test["mAP"],
    )
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return result


def write_result_csv(rows):
    columns = ["scenario", "snr_group", "snr_levels", "masking", "best_epoch", "epochs_run", *metric_names]
    with open(run.result_file, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow({**row, "snr_levels": "|".join(str(s) for s in row["snr_levels"]), **row["test"]})


separator.to(DEVICE).freeze()

scenarios = [
    (group, levels, masking) for group, levels in CONFIG["snr_groups"].items() for masking in CONFIG["maskings"]
]
logger.info("train beat with frcrn masking on %s | %d scenarios", MIXTURE_KEY, len(scenarios))

results = [
    run_scenario(index, len(scenarios), *scenario)
    for index, scenario in enumerate(tqdm(scenarios, desc="scenarios", unit="run"), start=1)
]
write_result_csv(results)

header = f"{'scenario':<16}" + "".join(f"{m:>18}" for m in metric_names)
print(header)
print("-" * len(header))
for _row in results:
    print(f"{_row['scenario']:<16}" + "".join(f"{_row['test'][m]:>18.4f}" for m in metric_names))
logger.info("result saved to %s", run.result_file)

In [ ]:
def show_table(title, rows):
    width = max(len(str(key)) for key, _ in rows) + 2
    print(f"\n{title}\n" + "=" * (width + 40))
    for key, value in rows:
        print(f"{key!s:<{width}}{value}")


def millions(module):
    return sum(p.numel() for p in module.parameters()) / 1e6


beat_config = beat_pretrained.config
frcrn_model_config = separator.config
with torch.no_grad():
    dummy = torch.zeros(1, int(CONFIG["clip_seconds"] * CONFIG["sample_rate"]))
    fbank = beat_pretrained.frontend(dummy)
    features = beat_pretrained.backbone(fbank)
    noise_estimate = separator.cpu()(dummy)

show_table(
    "Pipeline",
    [
        ("with masking", "mixture -> FRCRN (noise estimate) -> Fbank -> BEATs -> mean-pool -> Linear -> logits"),
        ("input waveform", tuple(dummy.shape)),
        ("FRCRN noise estimate", tuple(noise_estimate.shape)),
        ("Fbank", tuple(fbank.shape)),
        ("BEATs patch features", tuple(features.shape)),
        ("logits", (1, beat_config.num_classes)),
    ],
)

show_table(
    "FRCRN architecture",
    [
        *frcrn_model_config.to_dict().items(),
        ("stft bins", frcrn_model_config.num_bins),
        ("cascaded UNets", 2),
        ("encoder / decoder blocks per UNet", len(separator.network.unet.encoders)),
        ("channels", 128),
        ("FSMN order", 20),
        ("mask", "cIRM, tanh(u1) + tanh(u2(u1))"),
        ("parameters (M)", f"{millions(separator):.2f}"),
    ],
)

show_table(
    "BEATs architecture",
    [
        *beat_config.to_dict().items(),
        ("patch tokens N (clip)", features.shape[1]),
        ("backbone parameters (M)", f"{millions(beat_pretrained.backbone):.2f}"),
        ("head parameters (M)", f"{millions(beat_pretrained.head):.3f}"),
        ("total parameters (M)", f"{millions(beat_pretrained):.2f}"),
    ],
)

show_table("Frontend", list(beat_yaml["frontend"].items()))
show_table("BEATs training hyperparameters", list(CONFIG["beat"].to_dict().items()))
show_table(
    "Loss",
    [
        ("BEATs", f"CrossEntropy(label_smoothing={CONFIG['beat'].label_smoothing})"),
        ("selection metric", "validation macro_f1 (metrics.ClassificationMetrics)"),
    ],
)
show_table(
    "Experiment grid",
    [
        ("snr groups", CONFIG["snr_groups"]),
        ("masking", CONFIG["maskings"]),
        ("scenarios", len(CONFIG["snr_groups"]) * len(CONFIG["maskings"])),
        ("outputs", f"{run.log_file} | {run.checkpoint_dir} | {run.result_dir}"),
    ],
)

In [ ]:
from IPython.display import Image, display

from src.utils.plots import plot_frcrn_curve, plot_loss_curves, plot_map_curves, plot_test_metrics

snr_group_names = list(CONFIG["snr_groups"])
saved_results = {
    path.stem: json.loads(path.read_text(encoding="utf-8")) for path in sorted(run.history_dir.glob("*_*mask.json"))
}

figures = [
    plot_map_curves(saved_results, snr_group_names, run.figure_dir / "map_train_val.png"),
    plot_loss_curves(saved_results, snr_group_names, run.figure_dir / "train_loss.png"),
    plot_test_metrics(saved_results, snr_group_names, run.figure_dir / "test_metrics.png"),
]
for _figure in figures:
    logger.info("figure saved: %s", _figure)
    display(Image(filename=str(_figure)))